In [ ]:
%load_ext autoreload
%autoreload 2

## Training and Validation for Torchvision Maskrcnn on Volpy Data 
Tutorial for training and valdiation Maskrcnn model

@authors: Changjia Cai, Erik Thompson, and Manuel Paez

Date Created: August 28th, 2024

Date Updated: July 7th, 2025

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from skimage.color import rgb2gray, gray2rgb
import torch
from torch.optim.lr_scheduler import CyclicLR
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from tqdm import tqdm

from config import Config
from model import get_model_instance_segmentation, mrcnn_inference, thresholded_predictions 
from neurons import perform_final_evaluation, train_validate
from visualize import apply_masks, draw_boxes, vp_load_image

#### Check if cuda is available

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

## Volpy Data, Training, and Validation Sets
There are 24 datasets in total, 3 types of different voltage imaging datasets recorded from mouse L1 cortex (L1), mouse hippocampus (HPC), and zebrafish tegmental area (TEG). Set up the image directories from https://zenodo.org/records/4515768 as follows:

    volpy_training_data/
        images/
            HPC.29.04.npz
            ...
        masks/
            HPC.29.04_mask.npz
            ...      

### Config 
The base configuration class `Config` from `config.py` contains the variables necessary for training and validation on the VolPy dataset. By default it reads training data from `caiman_datadir()/volpy_training_data` and writes checkpoints to `caiman_datadir()/model`. CaImAn's data directory can be changed with `CAIMAN_DATA`; the two VolPy locations can also be overridden independently with `CAIMAN_VOLPY_TRAINING_DATA` and `CAIMAN_VOLPY_MODEL_DIR`.

For the Flatiron cluster layout used for this project, set these before starting Jupyter:

    export CAIMAN_DATA=/mnt/home/mpaez/caiman_data
    export CAIMAN_VOLPY_TRAINING_DATA=/mnt/home/mpaez/ceph/volpy_training_data
    export CAIMAN_VOLPY_MODEL_DIR=/mnt/home/mpaez/ceph/volpy_models/cai-pytorch-6

Training saves `mrcnn_latest.pt` and `volpy_train_history.pt` after every completed epoch, periodic `mrcnn_epoch_N.pt` snapshots according to `SAVE_FREQ`, and an epoch-numbered checkpoint for the final epoch. Existing training artifacts are protected by default; select a new `CAIMAN_VOLPY_MODEL_DIR` for a new run, or set `config.ALLOW_OVERWRITE = True` deliberately.

    Config: 
        # Paths
        DATA_DIR = caiman_datadir()/volpy_training_data
        MODEL_SAVE_DIR = caiman_datadir()/model/cai-pytorch-6

        # Model and Training Hyperparameters
        NUM_CLASSES = 1 + 1  # Background + Neuron
        BATCH_SIZE = 2 
        NUM_EPOCHS = 100
        MAX_LR = 0.005
        BASE_LR = 0.000001
        STEP_SIZE_UP = 3
        STEP_SIZE_DOWN = 7

        # Data Loading, Splitting, and Inference
        # IMAGES_PER_GPU = 2
        RANDOM_SPLIT = False # True for random split, False for fixed split from map below.
        NUM_TEST_RANDOM = 8
        NUM_TORCH_WORKERS = 4
        RANDOM_SEED = 42
        DATASET_REGION_MAP = {
            'HPC': [1],
            'L1': [12, 13, 14],
            'TEG': [22],
            'Train': [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 16, 17, 18, 19, 20, 21, 23]
        }

        INFERENCE_THRESHOLD = 0.5

        # Logging and Saving Frequency
        PRINT_FREQ = 1 
        SAVE_FREQ = 5

### Dataset 

Building on the standard `torch.utils.data.Dataset`class, we use 'NeuronsDataset' from neurons.py. The  `__getitem__` method of this class should return an `image` and a `target` dictionary delineating the different objects (box/mask) in the image:

    image: torchvision.tv_tensors.Image of shape [3, H, W]: can be a pure tensor, or a PIL Image of size (H, W)
    target: a dict containing the following keys
        masks : torchvision uint8 binary masks for each object (N,H,W) (N masks)
        boxes (bounding boxes)  (nx4)
        labels (int) label for each bounding box (note 0 is background, so if you have no bg, start with 1)
        image_id (int) unique image id
        area (float) area of bounding box 
        iscrowd (uint8) instances with `iscrowd=True` will be ignored during evaluation 

The 'data_transform' function from neurons.py should return a training pipeline for training. 
For windows, set the number of workers to 0. Otherwise, set to 4+

#### Dataset Split
The dataset split can be randomized split (such that each of L1, HPC, and TEG has a validation and training set) or can be set fixed. 

#### Data Transforms
From 'utils.py':

        transforms.append(T.RandomHorizontalFlip(p=0.5))
        transforms.append(T.RandomVerticalFlip(p=0.5))
        transforms.append(T.RandomApply([T.RandomRotation(degrees=(-5, 5))], p=0.5))
        transforms.append(T.ColorJitter(brightness=0.5,
                                        contrast=0.5,
                                        saturation=0.5,
                                        hue=0))
        transforms.append(T.GaussianBlur(kernel_size=(5, 5), sigma=(0.001, 0.3)))
        transforms.append(T.SanitizeBoundingBoxes(min_size=2))

#### Additional Training Setup:
Validation indices: [1, 12, 13, 14, 22]

Training indices: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 16, 17, 18, 19, 20, 21, 23]

In [ ]:
config = Config()
config.display()

### Model
We use a model pre-trained on the COCO dataset, and fine-tune the last layer. 

### Optimizer + lr scheduler
We use the SGD optimizer and a CycleLR scheduler 

In [ ]:
# Model construction, optimizer, scheduler, and deterministic sampling are
# centralized in train_validate() so the notebook and CLI cannot drift.

### Training the Network

In [ ]:
model, history = train_validate(config, plot_results=False)
all_train_losses = history['train_loss']
all_val_losses = history['val_loss']
all_lrs = history['lr']

### Plot Loss Function 

In [ ]:
plt.plot(np.array(all_train_losses), color='blue', marker='.', label='train')
plt.plot(np.array(all_val_losses), color='red', marker='.', label='validation')
plt.legend()
plt.xlabel('epoch')
plt.ylabel('net loss')
plt.title('loss function across different epochs')
plt.grid()

### Plot Learning Rate vs Epoch

In [ ]:
plt.plot(all_lrs, marker='.')
plt.xlabel('epoch')
plt.ylabel('learning rate')
plt.title('learning rate across different epochs')

### Inference using Test Set

For this, we will compute F1 scores for all datasets

#### Perform_final_evaluation 
From 'neurons.py', runs inference on the validtion set, calculates F1 scores for each region (i.e. HPC, L1, TEG), and reports the results. 

Note: Aim for 74 % >

In [ ]:
perform_final_evaluation(model, config, device, plot_results=True)